# Ruzivo AI ? Qwen2.5-1.5B QLoRA Fine-Tuning v7 (Complete 10k Corpus)
### Version: v7 (100% Book Coverage + Conversational Fluency + Anti-Hallucination)

- **Dataset:** 10,471 instruction-response pairs covering 19 Shona educational books.
- **Architecture:** Qwen/Qwen2.5-1.5B with 4-bit QLoRA on all linear projections (1.20% parameter budget).
- **Stops:** Strict <|im_end|> token handling to eliminate repetition loops.

In [ ]:
# 1. Install required packages
!pip install -q -U transformers datasets trl peft bitsandbytes accelerate

In [ ]:
# 2. Load Tokenizer & Quantized Base Model
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

BASE_MODEL = "Qwen/Qwen2.5-1.5B"
OUTPUT_DIR = "./ruzivo-llm-v7"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
print("Base model loaded in 4-bit NF4!")

In [ ]:
# 3. Configure LoRA on all 7 Linear Projections
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 4. Load the 10,471 Shona Instruction Dataset
DATASET_FILE = "shona_instructions_v7_complete.jsonl"
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
print(f"Loaded training dataset: {len(dataset):,} instruction samples!")

In [ ]:
# 5. Train v7 Adapter (2 Full Epochs over 10k samples)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=50,
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args
)

print("Starting v7 Training...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model successfully trained and saved to {OUTPUT_DIR}!")

In [ ]:
# 6. Package & Download v7
!zip -r ruzivo-llm-v7.zip ./ruzivo-llm-v7
print("Zip archive ruzivo-llm-v7.zip created! You can now download it.")

In [ ]:
# 7. Launch Live GPU Server with Token Stop & Anti-Repetition Guardrails
!pip install -q fastapi uvicorn pydantic
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import threading, uvicorn, urllib.request, torch

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

class ChatRequest(BaseModel):
    message: str

@app.post('/api/chat')
def chat(req: ChatRequest):
    query = req.message.strip()
    
    system_msg = (
        "Iwe uri Ruzivo, mubatsiri wehungwaru wekunyora nekutaura muChiShona chete. "
        "Pindura mibvunzo zvizere uye nechokwadi. "
        "Kana usingazivi mhinduro yacho, taura pachena kuti 'Handizivi'."
    )
    messages = [{"role": "system", "content": system_msg}, {"role": "user", "content": query}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>') or 151645
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=90,
            repetition_penalty=1.25,
            do_sample=False,
            eos_token_id=[im_end_id, tokenizer.eos_token_id],
            pad_token_id=tokenizer.pad_token_id
        )
    
    gen_ids = outputs[0][inputs['input_ids'].shape[1]:]
    reply = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return {'response': reply, 'context': None}

threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000), daemon=True).start()
!npm install -g localtunnel > /dev/null 2>&1
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f'=== PASSWORD (IP): {ip} ===')
!npx localtunnel --port 8000
